# **Deskripsi Dataset**

Dataset yang digunakan merupakan gambaran tentang mahasiswa yang terdaftar dalam 
berbagai program sarjana yang ditawarkan oleh perguruan tinggi. Data ini mencakup data 
demografis, faktor sosial-ekonomi, dan informasi kinerja akademik yang dapat digunakan 
untuk menganalisis faktor-faktor yang mungkin memprediksi tingkat kesuksesan akademik 
mahasiswa.

# **Requirements and Config**

In [2]:
# !pip install seaborn matplotlib numpy pandas

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.getcwd())
from allyouneed.tree import DecisionTreeClassifier
from allyouneed.model_selection import StratifiedKFold
from allyouneed.metrics import Accuracy, F1Score
from allyouneed.preprocessing import OneHotEncoder, LabelEncoder
from allyouneed.feature_selection import ForwardFeatureSelection, BackwardFeatureElimination

import warnings
warnings.filterwarnings("ignore")


pd.set_option('display.max_columns', None)

class settings:
    DATA_DIR = "../dataset/"
    TRAIN_FILE = "train.csv"
    TEST_FILE = "test.csv"
    SUBMISSION_FILE = "sample_submission.csv"
    INDEX_COL = "Student_ID"
    SEED = 42
    TARGET = "Target"
    TRAIN_PATH = DATA_DIR + TRAIN_FILE
    TEST_PATH = DATA_DIR + TEST_FILE
    SUBMISSION_PATH = DATA_DIR + SUBMISSION_FILE

In [4]:
train = pd.read_csv(settings.TRAIN_PATH, index_col=settings.INDEX_COL)
test = pd.read_csv(settings.TEST_PATH, index_col=settings.INDEX_COL)
submission = pd.read_csv(settings.SUBMISSION_PATH, index_col=settings.INDEX_COL)
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3096 entries, 3743 to 3303
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  3096 non-null   int64  
 1   Application mode                                3096 non-null   int64  
 2   Application order                               3096 non-null   int64  
 3   Course                                          3096 non-null   int64  
 4   Daytime/evening attendance	                     3096 non-null   int64  
 5   Previous qualification                          3096 non-null   int64  
 6   Previous qualification (grade)                  3096 non-null   float64
 7   Nacionality                                     3096 non-null   int64  
 8   Mother's qualification                          3096 non-null   int64  
 9   Father's qualification                     

# **Exploratory Data Analysis**

# **Preprocessing**

## **Data Cleaning**

In [5]:
def drop_columns(df):
    col = []
    return df.drop(columns=col)

## **Data Transformation**

In [6]:
label_encoder = LabelEncoder()
label_encoder.fit(train[settings.TARGET])

## **Feature Selection**

## **Dimensionality Reduction**

## **Pipeline**

In [7]:
class Preprocessing:
    def __init__(self, categorical_cols=None):
        self.categorical_cols = categorical_cols or []
        self.ohe = OneHotEncoder(columns=categorical_cols, dtype=float)
        self.numeric_cols = []
        self.fitted = False

    def fit(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.numeric_cols = [col for col in self.numeric_cols if col not in self.categorical_cols]
        self.ohe.fit(X)
        self.fitted = True
        return self

    def transform(self, X):
        if not self.fitted:
            raise ValueError("Preprocessing must be fitted before transform()")

        if not isinstance(X, pd.DataFrame):
            raise ValueError("X must be a pandas DataFrame")

        # Numeric features (tetap DataFrame)
        X_num = X[self.numeric_cols].astype(float)

        # Categorical features (one-hot encoded)
        X_cat_array = self.ohe.transform(X)
        X_cat = pd.DataFrame(
            X_cat_array,
            columns=self.ohe.feature_names_,
            index=X.index
        )

        # Combine dan return DataFrame
        result = pd.concat([X_num, X_cat], axis=1)
        return result

    def fit_transform(self, X):
        return self.fit(X).transform(X)


CATEGORICAL_COLS = [
    "Marital status",
    "Course",
    "Previous qualification",
]

DROPPED_COLS = [
    
]


preprocessor = Preprocessing(categorical_cols=CATEGORICAL_COLS)
X_train, y_train = train.drop(columns=[settings.TARGET]), label_encoder.transform(train[settings.TARGET])
X_train = preprocessor.fit_transform(X_train)
y_train = label_encoder.transform(train[settings.TARGET])

ffs = ForwardFeatureSelection(
    estimator=DecisionTreeClassifier(random_state=settings.SEED),
    n_features_to_select=10,
    scoring=F1Score(average='macro'),
    verbose=True,
    feature_names=X_train.columns.tolist(),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.SEED),
    n_jobs=10
)


In [8]:
# ffs.fit(X_train, y_train)
# ffs.selected_features_

# **Modeling and Validation**

## **Metrics**

In [9]:
# ============================================================
# Utility: Confusion Matrix
# ============================================================
class ConfusionMatrix:
    """
    Computes confusion matrix using NumPy.
    """

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        labels = np.unique(np.concatenate([y_true, y_pred]))
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        
        cm = np.zeros((len(labels), len(labels)), dtype=int)

        for t, p in zip(y_true, y_pred):
            cm[label_to_idx[t], label_to_idx[p]] += 1
        
        return cm, labels

# ============================================================
# Metric Collection (Run multiple metrics at once)
# ============================================================
class MetricCollection:
    """
    Utility to compute multiple metrics at once.
    """

    def __init__(self, metrics: dict):
        """
        metrics: dict[name -> Metric instance]
        """
        self.metrics = metrics

    def __call__(self, y_true, y_pred):
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}


## **Modeling**

In [10]:
X_train_selected = X_train[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.SEED)

models = {
    'DT': DecisionTreeClassifier(random_state=settings.SEED),
}

metrics = MetricCollection({
    'accuracy': Accuracy(),
    'f1_macro': F1Score(average='macro'),
    'f1_micro': F1Score(average='micro'),
    'f1_weighted': F1Score(average='weighted')  
})

all_results = {} 

In [12]:


for name, model in models.items():
    print(f"Training model: {name}")
    fold_results = []  # store metric dict for each fold

    for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[settings.TARGET])):
        
        # -------------------------
        # Split data
        # -------------------------
        X_train = train.drop(columns=[settings.TARGET]).iloc[train_idx]
        y_train_raw = train[settings.TARGET].iloc[train_idx].values
        y_train = label_encoder.transform(y_train_raw)

        X_val = train.drop(columns=[settings.TARGET]).iloc[val_idx]
        y_val_raw = train[settings.TARGET].iloc[val_idx].values
        y_val = label_encoder.transform(y_val_raw)

        # -------------------------
        # Preprocessing
        # -------------------------
        train_data = preprocessor.fit_transform(X_train)
        val_data = preprocessor.transform(X_val)

        # -------------------------
        # Training
        # -------------------------
        model.fit(train_data, y_train)

        # -------------------------
        # Validation
        # -------------------------
        val_preds = model.predict(val_data)

        # -------------------------
        # Metrics
        # -------------------------
        results = metrics(y_val, val_preds)
        fold_results.append(results)

        print(f"Fold {fold + 1} results: {results}")

    # -------------------------
    # Aggregate across folds
    # -------------------------
    df_results = pd.DataFrame(fold_results)
    model_mean = df_results.mean().to_dict()

    all_results[name] = {
        "per_fold": fold_results,
        "mean": model_mean
    }

    print(f"\nAggregated results for model {name}:")
    print(df_results)
    print("Mean metrics:", model_mean)
    print("=" * 60)


Training model: DT
Fold 1 results: {'accuracy': 0.6666666666666666, 'f1_macro': 0.6037784931551394, 'f1_micro': 0.6666666666666666, 'f1_weighted': 0.6680952221620577}


KeyboardInterrupt: 

# ✅**Final Submission**

In [13]:
train_df = pd.read_csv(settings.TRAIN_PATH)
test_df  = pd.read_csv(settings.TEST_PATH)

X_train_full = train_df.drop(columns=[settings.TARGET])
y_train_full = label_encoder.transform(train_df[settings.TARGET].values)

X_train_full_proc = preprocessor.fit_transform(X_train_full).astype(float)
X_train_full_proc = X_train_full_proc[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

model = models['DT']
model.fit(X_train_full_proc, y_train_full)

print("Final model trained.")

X_test_proc = preprocessor.transform(test_df).astype(float)
X_test_proc = X_test_proc[['Curricular units 2nd sem (approved)', 'Course__9773', 'Curricular units 2nd sem (credited)', 'Debtor', 'Tuition fees up to date', 'Course__9500', 'Course__171', 'Course__9119']]

test_preds_int = model.predict(X_test_proc)
test_preds = label_encoder.inverse_transform(test_preds_int)

submission = pd.read_csv(settings.SUBMISSION_PATH)

submission[settings.TARGET] = test_preds
submission.to_csv("sub-5.csv", index=False)

print("\nSubmission saved to sub-5.csv")

Final model trained.

Submission saved to sub-5.csv
